# 野生蓝莓产量预测分析## 步骤 3：聚类分析 (K-Means)

In [ ]:
import pandas as pdimport numpy as npimport sys, ossys.path.insert(0, os.path.join(os.getcwd(), ".."))from src.config import *from src.data.loader import load_trainfrom src.data.preprocessor import DataPreprocessorfrom src.models.clustering import BlueberryClusteringfrom src.features.engineering import apply_pcafrom src.visualization.plots import *train = load_train()preprocessor = DataPreprocessor()df_clean = preprocessor.clean(train)X, y = preprocessor.split_features_target(df_clean)X_scaled = preprocessor.scale(X, fit=True)print(f"X_scaled shape: {X_scaled.shape}")

## 3.1 确定最优K值（肘部法则 + 轮廓系数）

In [ ]:
cluster = BlueberryClustering()results = cluster.find_optimal_k(X_scaled, k_range=range(2, 11))print(f"Optimal K (silhouette): {results['best_k']}")print(f"\nSilhouette scores:")for k, s in zip(results['k_range'], results['silhouette_scores']):    print(f"  K={k}: {s:.4f}")print(f"\nDavies-Bouldin scores:")for k, s in zip(results['k_range'], results['davies_bouldin_scores']):    print(f"  K={k}: {s:.4f}")

In [ ]:
plot_elbow(results["inertias"], results["k_range"], results["best_k"])

## 3.2 执行聚类

In [ ]:
cluster.fit(X_scaled, k=results["best_k"])stats = cluster.get_cluster_stats(X_scaled)print(f"Cluster count: {np.bincount(cluster.labels)}")print("\nCluster means:")print(stats)

## 3.3 PCA降维可视化

In [ ]:
X_pca, pca, explained = apply_pca(X_scaled, n_components=2)print(f"PCA explained variance: {explained}")plot_cluster_scatter(X_pca.values, cluster.get_predictions())

## 3.4 聚类群组解读

In [ ]:
# 分析各聚类与原特征的关系X_with_labels = X.copy()X_with_labels["cluster"] = cluster.labelsprint("Cluster mean values (original scale):")print(X_with_labels.groupby("cluster").mean())print("\nCluster yield correlation:")for c in sorted(X_with_labels["cluster"].unique()):    cluster_y = y[X_with_labels["cluster"] == c]    print(f"  Cluster {c}: mean yield = {cluster_y.mean():.2f}, std = {cluster_y.std():.2f}")